# CommuteCast — Stage 3: Probability, Conditional Probability, Expectation & Bayes' Theorem

## Unit-1 Conceptual Foundations:
- **Basic Empirical Probability**: $P(A) = \frac{\text{count}(A)}{N}$
- **Conditional Probability**: $P(A \mid B) = \frac{\text{count}(A \cap B)}{\text{count}(B)}$
- **Empirical Expectation**: Estimating $E[X]$ and $E[X \mid B]$ using subpopulation sample means $\bar{x}_B$.
- **Bayes' Theorem**: $P(A \mid B) = \frac{P(B \mid A) \cdot P(A)}{P(B)}$
- **Statistical Independence**: Comparing $P(A \mid B)$ with $P(A)$ without claiming causality.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to sys.path
sys.path.insert(0, os.path.abspath('..'))
from src.probability import (
    calculate_congestion_probabilities,
    calculate_rain_probabilities,
    calculate_rain_conditional_probabilities,
    calculate_expectation,
    calculate_peak_hour_analysis,
    calculate_rain_and_peak_joint_analysis,
    calculate_temporal_conditional_analysis,
    evaluate_statistical_independence,
    CORE_PEAK_HOURS
)
from src.bayes import bayes_probability, verify_bayes_with_empirical_probability, format_bayes_report
from src.visualizations import (
    plot_stage3_prior_probabilities,
    plot_stage3_rain_conditional_high,
    plot_stage3_rain_expected_traffic,
    plot_stage3_peak_conditional_high,
    plot_stage3_weekday_conditional_high,
    plot_stage3_rain_peak_conditional_high,
    generate_stage3_dashboard
)

# Load cleaned dataset
df = pd.read_csv('../data/preprocessed_traffic_volume.csv')
print(f'Dataset loaded: {df.shape[0]:,} observations, {df.shape[1]} columns.')

## 1. Basic Congestion & Rain Probabilities

$$
P(A) = \frac{\text{count}(A)}{N}
$$

Probability axioms require that mutually exclusive and collectively exhaustive event sets sum to $1.0$.

In [ ]:
cg_res = calculate_congestion_probabilities(df)
rain_res = calculate_rain_probabilities(df)

cg_table = pd.DataFrame({
    'Event': ['Low Congestion', 'Medium Congestion', 'High Congestion', 'Total / Sum'],
    'Count': [cg_res['count_low'], cg_res['count_medium'], cg_res['count_high'], cg_res['total_observations']],
    'Probability P(A)': [cg_res['P_Low'], cg_res['P_Medium'], cg_res['P_High'], cg_res['sum_probabilities']],
    'Percentage (%)': [cg_res['P_Low']*100, cg_res['P_Medium']*100, cg_res['P_High']*100, cg_res['sum_probabilities']*100]
})
display(cg_table)

rain_table = pd.DataFrame({
    'Event': ['Rain Present (rain=1)', 'No Rain (rain=0)', 'Total / Sum'],
    'Count': [rain_res['count_rain'], rain_res['count_norain'], rain_res['total_observations']],
    'Probability P(A)': [rain_res['P_Rain'], rain_res['P_NoRain'], rain_res['sum_probabilities']],
    'Percentage (%)': [rain_res['P_Rain']*100, rain_res['P_NoRain']*100, rain_res['sum_probabilities']*100]
})
display(rain_table)

## 2. Conditional Probability: Congestion Given Rain

$$
P(A \mid B) = \frac{\text{count}(A \cap B)}{\text{count}(B)}
$$

In [ ]:
rain_cond = calculate_rain_conditional_probabilities(df)
cond_table = pd.DataFrame({
    'Condition': ['Given Rain (rain=1)', 'Given No Rain (rain=0)'],
    'P(High | Condition)': [rain_cond['P_High_given_Rain'], rain_cond['P_High_given_NoRain']],
    'P(Medium | Condition)': [rain_cond['P_Medium_given_Rain'], rain_cond['P_Medium_given_NoRain']],
    'P(Low | Condition)': [rain_cond['P_Low_given_Rain'], rain_cond['P_Low_given_NoRain']],
    'Sum of Row': [
        rain_cond['P_High_given_Rain'] + rain_cond['P_Medium_given_Rain'] + rain_cond['P_Low_given_Rain'],
        rain_cond['P_High_given_NoRain'] + rain_cond['P_Medium_given_NoRain'] + rain_cond['P_Low_given_NoRain']
    ]
})
display(cond_table)

## 3. Empirical Expectations

$$
\hat{E}[X \mid B] = \bar{x}_B = \frac{1}{n_B} \sum_{i \in B} x_i
$$

In [ ]:
e_overall = calculate_expectation(df)
e_rain = calculate_expectation(df, condition=(df['rain'] == 1))
e_norain = calculate_expectation(df, condition=(df['rain'] == 0))
e_high = calculate_expectation(df, condition=(df['congestion'] == 'High'))

print(f'E(Traffic)                  : {e_overall:,.2f} veh/hr')
print(f'E(Traffic | Rain)           : {e_rain:,.2f} veh/hr')
print(f'E(Traffic | No Rain)        : {e_norain:,.2f} veh/hr')
print(f'E(Traffic | High Congestion): {e_high:,.2f} veh/hr')

## 4. Peak-Hour & Joint Rain+Peak Analysis

**Peak Hour Selection Justification**:
Examining the diurnal hourly curve from Stage 2 reveals two pronounced rush-hour peaks:
- Morning: Hours 7:00 & 8:00 AM ($P(\text{High}) > 62\%$)
- Evening: Hours 16:00 & 17:00 PM (4–6 PM, $P(\text{High}) > 65\%$)

Core Peak Hours = `[7, 8, 16, 17]`.

In [ ]:
peak_res = calculate_peak_hour_analysis(df)
joint_res = calculate_rain_and_peak_joint_analysis(df)

print(f"P(Peak Hour)            : {peak_res['P_Peak']:.4f} ({peak_res['count_peak']:,} / {len(df):,})")
print(f"P(High | Peak Hour)     : {peak_res['P_High_given_Peak']:.4f}")
print(f"P(High | Non-Peak Hour) : {peak_res['P_High_given_NonPeak']:.4f}")
print(f"E(Traffic | Peak Hour)  : {peak_res['E_Traffic_Peak']:,.2f} veh/hr")
print(f"E(Traffic | Non-Peak)   : {peak_res['E_Traffic_NonPeak']:,.2f} veh/hr")
print()
print(f"P(High | Rain, Peak Hour)   : {joint_res['P_High_given_Rain_Peak']:.4f} (n={joint_res['count_rain_peak']:,})")
print(f"P(High | No Rain, Peak Hour): {joint_res['P_High_given_NoRain_Peak']:.4f} (n={joint_res['count_norain_peak']:,})")
print(f"E(Traffic | Rain, Peak)     : {joint_res['E_Traffic_Rain_Peak']:,.2f} veh/hr")
print(f"E(Traffic | No Rain, Peak)  : {joint_res['E_Traffic_NoRain_Peak']:,.2f} veh/hr")

## 5. Temporal Conditions: Weekday vs Weekend & Holiday Analysis

In [ ]:
temp_res = calculate_temporal_conditional_analysis(df)

temp_table = pd.DataFrame({
    'Condition': ['Weekday', 'Weekend', 'Holiday', 'Non-Holiday'],
    'Observation Count': [temp_res['count_weekday'], temp_res['count_weekend'], temp_res['count_holiday'], temp_res['count_nonholiday']],
    'P(High | Condition)': [temp_res['P_High_given_Weekday'], temp_res['P_High_given_Weekend'], temp_res['P_High_given_Holiday'], temp_res['P_High_given_NonHoliday']],
    'E(Traffic | Condition)': [temp_res['E_Traffic_Weekday'], temp_res['E_Traffic_Weekend'], temp_res['E_Traffic_Holiday'], temp_res['E_Traffic_NonHoliday']]
})
display(temp_table)

## 6. Bayes' Theorem Implementation & Exact Empirical Verification

$$
P(A \mid B) = \frac{P(B \mid A) \cdot P(A)}{P(B)}
$$

In [ ]:
high_mask = (df['congestion'] == 'High')
rain_mask = (df['rain'] == 1)
peak_mask = df['hour'].isin(CORE_PEAK_HOURS)

v_rain = verify_bayes_with_empirical_probability(df, high_mask, rain_mask, 'High', 'Rain')
v_peak = verify_bayes_with_empirical_probability(df, high_mask, peak_mask, 'High', 'Peak')

bayes_table = pd.DataFrame({
    'Example': ['P(High | Rain)', 'P(High | Peak)'],
    'Prior P(A)': [v_rain['prior_P_A'], v_peak['prior_P_A']],
    'Marginal P(B)': [v_rain['marginal_P_B'], v_peak['marginal_P_B']],
    'Likelihood P(B|A)': [v_rain['likelihood_P_B_given_A'], v_peak['likelihood_P_B_given_A']],
    'Bayes Result': [v_rain['bayes_posterior'], v_peak['bayes_posterior']],
    'Direct Empirical': [v_rain['direct_empirical_posterior'], v_peak['direct_empirical_posterior']],
    'Absolute Difference': [v_rain['absolute_difference'], v_peak['absolute_difference']],
    'Exact Match?': [v_rain['is_verified'], v_peak['is_verified']]
})
display(bayes_table)

## 7. Statistical Independence Evaluation

Two events $A$ and $B$ are statistically independent if and only if:
$$P(A \mid B) = P(A)$$

> **Core Principle**: Statistical dependence does not imply causality.

In [ ]:
p_high = cg_res['P_High']
print('1. Rain vs High Congestion:')
print(evaluate_statistical_independence(p_high, rain_cond['P_High_given_Rain'], 'Rain'))
print()
print('2. Peak Hour vs High Congestion:')
print(evaluate_statistical_independence(p_high, peak_res['P_High_given_Peak'], 'Peak Hour'))
print()
print('3. Weekday vs High Congestion:')
print(evaluate_statistical_independence(p_high, temp_res['P_High_given_Weekday'], 'Weekday'))

## 8. Stage 3 Visualizations

In [ ]:
fig_dash = generate_stage3_dashboard(df)
plt.show()